# 🎓 StudentPulse ML — End-to-End Demo Notebook

This notebook walks through the complete StudentPulse ML pipeline:

1. **Data Loading & Preprocessing**
2. **Feature Engineering**
3. **Similarity Computation**
4. **Student Clustering**
5. **Recommendation Engine**
6. **Evaluation**
7. **Cold-Start Demo**

> Run from the `student-pulse/` directory: `jupyter notebook notebooks/student_pulse_ml.ipynb`

## 0. Setup

In [ ]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print('Environment ready.')

## 1. Data Loading & Preprocessing

In [ ]:
from ml.preprocessing import preprocess_all

data = preprocess_all()

students     = data['students']
opportunities = data['opportunities']
papers       = data['papers']

print(f'Students     : {len(students)} rows')
print(f'Opportunities: {len(opportunities)} rows')
print(f'Papers       : {len(papers)} rows')

In [ ]:
# Preview students
students[['student_id','name','department','cgpa','year_of_study','skills']].head()

In [ ]:
# CGPA distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(students['cgpa'].astype(float), bins=10, color='steelblue', edgecolor='white')
axes[0].set_title('CGPA Distribution')
axes[0].set_xlabel('CGPA')

dept_counts = students['department'].value_counts()
axes[1].bar(dept_counts.index, dept_counts.values, color='coral', edgecolor='white')
axes[1].set_title('Students by Department')
axes[1].tick_params(axis='x', rotation=30)

year_counts = students['year_of_study'].value_counts().sort_index()
axes[2].bar(year_counts.index.astype(str), year_counts.values, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Students by Year of Study')

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
from ml.feature_engineering import build_all_features

features = build_all_features(data)

print(f"Student  matrix : {features['student_matrix'].shape}")
print(f"Opp      matrix : {features['opp_matrix'].shape}")
print(f"Paper    matrix : {features['paper_matrix'].shape}")
print(f"Skill vocab size: {len(features['skill_vocab'])}")
print(f"\nSample skills in vocab: {features['skill_vocab'][:10]}")

## 3. Similarity Computation

In [ ]:
from ml.similarity import compute_all_similarities

sims = compute_all_similarities(
    features,
    students_df=students,
    opps_df=opportunities,
    papers_df=papers
)

for k, v in sims.items():
    print(f'{k:25s}  shape={str(v.shape):15s}  min={v.min():.3f}  max={v.max():.3f}  mean={v.mean():.3f}')

In [ ]:
# Visualise student-opportunity similarity heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(sims['opp_cosine'], aspect='auto', cmap='YlOrRd')
axes[0].set_title('Student ↔ Opportunity Cosine Similarity')
axes[0].set_xlabel('Opportunity Index')
axes[0].set_ylabel('Student Index')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(sims['paper_cosine'], aspect='auto', cmap='YlGnBu')
axes[1].set_title('Student ↔ Paper Cosine Similarity')
axes[1].set_xlabel('Paper Index')
axes[1].set_ylabel('Student Index')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

## 4. Student Clustering

In [ ]:
from ml.clustering import cluster_students, summarize_clusters, pca_2d

model, labels = cluster_students(features['student_matrix'])
summary = summarize_clusters(students, labels)
print(summary.to_string(index=False))

In [ ]:
# PCA 2D cluster visualisation
pca_df = pca_2d(features['student_matrix'], labels)

plt.figure(figsize=(8, 6))
n_clusters = len(set(labels))
colors = cm.Set1(np.linspace(0, 1, n_clusters))

for cid, color in zip(range(n_clusters), colors):
    mask = pca_df['cluster'] == cid
    plt.scatter(pca_df[mask]['x'], pca_df[mask]['y'],
                label=f'Cluster {cid}', color=color, s=80, edgecolors='white', linewidths=0.5)

# Annotate student names
for i, row in pca_df.iterrows():
    plt.annotate(students.iloc[i]['student_id'], (row['x'], row['y']),
                 fontsize=6, ha='center', va='bottom')

plt.title('Student Clusters (PCA 2D Projection)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Recommendation Engine

In [ ]:
from ml.recommendation_engine import RecommendationEngine

engine = RecommendationEngine()
engine.fit(data, features, sims, model, labels)
print('Engine fitted successfully.')

In [ ]:
# Recommend for student S001 (Aarav Sharma — Data Scientist)
result = engine.recommend_all('S001', top_k_opps=5, top_k_papers=5)

print(f"\n{'='*65}")
print(f"  Recommendations for {result.student_name} ({result.student_id})")
print(f"{'='*65}")

print('\n--- Top 5 Opportunities ---')
for i, r in enumerate(result.opportunities, 1):
    print(f'  {i}. [{r.score:.3f}] {r.title}')
    print(f'       Skills matched: {r.matched_skills}')

print('\n--- Top 5 Research Papers ---')
for i, r in enumerate(result.papers, 1):
    print(f'  {i}. [{r.score:.3f}] {r.title}')

In [ ]:
# Score breakdown for top opportunity
top_opp = result.opportunities[0]
print(engine.get_explanation('S001', top_opp.item_id, 'opportunity'))

In [ ]:
# Visualise score breakdown for top-5 opportunities
opps = result.opportunities[:5]
titles = [r.title[:30] for r in opps]
cosines  = [r.cosine_score  for r in opps]
overlaps = [r.skill_overlap for r in opps]
scores   = [r.score         for r in opps]

x = range(len(titles))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([xi - width for xi in x], cosines,  width, label='Cosine',        color='steelblue')
ax.bar([xi          for xi in x], overlaps, width, label='Skill Overlap', color='coral')
ax.bar([xi + width for xi in x], scores,   width, label='Final Score',   color='mediumseagreen')
ax.set_xticks(x)
ax.set_xticklabels(titles, rotation=20, ha='right')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title(f'Score Breakdown — {result.student_name}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Filtered recommendation (internships only, remote)
filtered = engine.recommend_opportunities('S001', top_k=5, filters={'type': 'internship', 'remote': True})
print('Remote Internships for S001:')
for r in filtered:
    print(f'  [{r.score:.3f}] {r.title}')

## 6. Evaluation

In [ ]:
from ml.evaluate import full_evaluation_report, print_report

report = full_evaluation_report(engine, data, k_values=[5, 10])
print_report(report)

In [ ]:
# Plot NDCG and Hit Rate across K
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ks = report['k'].tolist()

for ax, metric_prefix, title, color_pair in [
    (axes[0], 'ndcg',     'NDCG @ K',     ('steelblue', 'coral')),
    (axes[1], 'hit_rate', 'Hit Rate @ K', ('mediumseagreen', 'orchid')),
]:
    opp_col   = [f'opp_{metric_prefix}@{k}'   for k in ks]
    paper_col = [f'paper_{metric_prefix}@{k}' for k in ks]

    opp_vals   = [report.loc[report['k']==k, c].values[0] if c in report.columns else 0 for k, c in zip(ks, opp_col)]
    paper_vals = [report.loc[report['k']==k, c].values[0] if c in report.columns else 0 for k, c in zip(ks, paper_col)]

    ax.plot(ks, opp_vals,   marker='o', label='Opportunities', color=color_pair[0])
    ax.plot(ks, paper_vals, marker='s', label='Papers',        color=color_pair[1])
    ax.set_title(title)
    ax.set_xlabel('K')
    ax.set_xticks(ks)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Cold-Start Demo (New Student)

In [ ]:
cold_result = engine.recommend_for_new_student(
    skills=['python', 'machine learning', 'sql'],
    interests=['data science', 'ai'],
    cgpa=8.2,
    top_k=5
)

print('Cold-Start Recommendations (New Student):')
print('\n--- Top Opportunities ---')
for r in cold_result.opportunities:
    print(f'  [{r.score:.3f}] {r.title}')

print('\n--- Top Papers ---')
for r in cold_result.papers:
    print(f'  [{r.score:.3f}] {r.title}')

## 8. Save & Load Engine

Use `train.py` to persist the fully trained engine to disk.

In [ ]:
from ml.train import train, load_engine

# Full training run (also saves artefacts)
engine = train()

# Load back
engine2 = load_engine()
r = engine2.recommend_all('S001')
print(f'Loaded engine — top opp: {r.opportunities[0].title}')